# 1 - HyperParameters Optimization

# 1 - 1 - RandomizedSearchCV

In [ ]:
param_grid = {
    "regressor__n_estimators": [100, 200, 300, 500],
    "regressor__max_depth": [None, 10, 20, 30],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4],
    "regressor__max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=random_forest_model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=1
)

random_search.fit(X_train,y_train)

print(random_search.best_params_)

print(-random_search.best_score_)

best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

# 1 - 2 - GridSearchCV

In [ ]:
param_grid = {
    "regressor__n_estimators": [250, 300, 350, 400],
    "regressor__max_depth": [25, 30, 35, 40],
    "regressor__min_samples_split": [8, 10, 12],
    "regressor__min_samples_leaf": [1, 2, 3],
    "regressor__max_features": ["sqrt"]
}

grid_search = GridSearchCV(
    random_forest_model,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=1
)

grid_search.fit(X_train,y_train)

print(grid_search.best_params_)

print(-grid_search.best_score_)

best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

# 2 - Tunning the Model

In [ ]:
random_forest_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",RandomForestRegressor(
        n_estimators=250,
        max_depth=35,
        min_samples_split=10,
        min_samples_leaf=1,
        max_features='sqrt'
    ))
])

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

for metric, scoring_name in scoring.items():
        scores = cross_val_score(
            random_forest_model, X_train, y_train,
            cv=5,
            scoring=scoring_name
        )

        if scoring_name.startswith("neg_"):
            scores = -scores

        x_validation_df.loc[f"Mean {metric}"] = scores.mean()
        x_validation_df.loc[f"{metric}_STD"] = scores.std()

random_forest_model.fit(X_train,y_train)

    predictions = random_forest_model.predict(X_test)
    test_evaluations = [
        mean_absolute_error(y_test,predictions),
        mean_squared_error(y_test,predictions),
        np.sqrt(mean_squared_error(y_test,predictions)),
        round(r2_score(y_test,predictions)*100,2)
    ]
    testing_df = test_evaluations

    predictions = random_forest_model.predict(X_train)
    train_evaluations = [
        mean_absolute_error(y_train,predictions),
        mean_squared_error(y_train,predictions),
        np.sqrt(mean_squared_error(y_train,predictions)),
        round(r2_score(y_train,predictions)*100,2)
    ]
    training_df = train_evaluations

print("Cross Validation :")
print(x_validation_df)
print("="*40)
print("Testing Results")
print(testing_df.sort_values(by="R2(%)",ascending=False))
print("="*40)
print("Training Results")
print(training_df.sort_values(by="R2(%)",ascending=False))
print("\n")